# Training Dynamics: Gradient Descent Optimizers

**Learning Objectives:**
- Understand how Batch Gradient Descent works and its limitations
- Learn why sharp minima lead to overfitting
- Discover how Stochastic Gradient Descent (SGD) prevents overfitting through noise
- Master Adam optimizer and when to use each approach

**Estimated Time:** 40-50 minutes

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. The Optimization Challenge

Training a neural network means finding parameters that minimize the loss function. But **how** we compute and apply gradients dramatically affects:
- Training speed
- Final model quality
- Generalization to new data

We'll explore three fundamental approaches:
1. **Batch Gradient Descent (GD)**: Use entire dataset per update
2. **Stochastic Gradient Descent (SGD)**: Use small mini-batches
3. **Adam**: Adaptive learning with momentum

Let's start with the most basic approach.

## 2. Batch Gradient Descent (GD)

### How It Works

Batch GD computes the gradient using **all training samples** before making a single update:

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} \frac{1}{N} \sum_{i=1}^{N} L(x_i, y_i; \theta)$$

Where:
- $N$ = total number of training samples
- $\eta$ = learning rate
- The gradient is computed over the **entire dataset**

### Characteristics

✅ **Exact gradient**: No approximation, uses all data  
✅ **Smooth convergence**: Deterministic, reproducible path  
✅ **Guaranteed descent**: Each step reduces loss (with proper learning rate)

❌ **Computationally expensive**: Must process all data per update  
❌ **Doesn't scale**: Impractical for large datasets  
❌ **Memory intensive**: Needs to load entire dataset

### PyTorch Implementation

In PyTorch, Batch GD means using a batch size equal to your entire dataset:

In [ ]:
# Simple model for demonstration
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

# Create synthetic dataset
X_train = torch.randn(1000, 10)  # 1000 samples
y_train = torch.randn(1000, 1)

# Batch GD: batch_size = entire dataset
train_loader_gd = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=len(X_train),  # Full batch!
    shuffle=False
)

# Optimizer
optimizer_gd = optim.SGD(model.parameters(), lr=0.01)

# Training loop for Batch GD
criterion = nn.MSELoss()
losses_gd = []

for epoch in range(50):
    for X_batch, y_batch in train_loader_gd:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward pass (using entire dataset)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer_gd.zero_grad()
        loss.backward()
        optimizer_gd.step()
        
        losses_gd.append(loss.item())

print(f"Batch GD - Final loss: {losses_gd[-1]:.4f}")

### Visualizing Batch GD Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses_gd, linewidth=2, color='blue')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Batch Gradient Descent: Smooth, Deterministic Convergence', fontsize=14, weight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: Very smooth decrease - no noise, exact gradient each step")

## 3. The Problem: Sharp Minima and Overfitting

### Understanding Sharp vs Flat Minima

Not all minima are equal! The loss landscape can have:

#### 🔴 Sharp Minima (Bad - Overfitting)
- **Steep walls**: Small parameter changes → Large loss changes
- **Memorization**: Model fits training data too precisely
- **Poor generalization**: Doesn't work well on new data
- **Fragile**: Sensitive to perturbations

#### 🟢 Flat Minima (Good - Generalization)
- **Wide valleys**: Parameters can vary without hurting loss
- **Robust patterns**: Model learned general features
- **Good generalization**: Works well on new data
- **Stable**: Insensitive to small changes

### Why Batch GD Falls Into Sharp Minima

Batch GD follows the exact gradient, which means it:
1. Takes the **most direct path** to the nearest minimum
2. Has **no exploration** - deterministic trajectory
3. **Cannot escape** once it reaches a minimum (gradient = 0)
4. Often finds sharp minima because they're **easier to fall into** (steep slopes guide gradient)

**Result**: The model overfits to the training data!

### Conceptual Illustration

```
Sharp Minimum (Overfit):        Flat Minimum (Good Generalization):

Loss                             Loss
  |    /\                          |    
  |   /  \                         |   /‾‾‾‾‾\
  |  /    \                        |  /       \
  | /      \                       | /         \
  |/   GD   \                      |/    SGD    \
  |   falls  \                     |   prefers  \
  |    here!  \                    |    here!    \
  +--------------> Parameters      +---------------> Parameters
  
  Steep walls = Overfit           Wide valley = Generalizes
```

### Additional Problem: Saddle Points

In high-dimensional spaces (neural networks have millions of parameters!), **saddle points** are common:
- Gradient ≈ 0, but it's not a minimum
- Batch GD slows down dramatically
- Can stall optimization for many iterations

**Solution?** Add noise to escape sharp minima and saddle points!

## 4. Stochastic Gradient Descent (SGD): The Power of Noise

### How It Works

Instead of using all data, SGD uses **small random mini-batches**:

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} \frac{1}{B} \sum_{i \in \mathcal{B}_t} L(x_i, y_i; \theta)$$

Where:
- $\mathcal{B}_t$ = random mini-batch at step $t$ (typically 32-256 samples)
- $B$ = batch size (much smaller than $N$)

### Key Difference: Noisy Gradients

Each mini-batch gives a **different gradient estimate** → introduces **beneficial noise**

### Why Noise Helps

1. **Escapes Sharp Minima**
   - Noise "kicks" optimizer out of steep valleys
   - Prevents overfitting by not settling into overfit solutions
   - Explores the loss landscape

2. **Navigates Saddle Points**
   - Random perturbations break symmetry
   - Helps escape plateaus faster

3. **Finds Flat Minima**
   - Tends to settle in wide valleys
   - These represent better generalization

4. **Implicit Regularization**
   - Noise acts like regularization
   - Prevents model from fitting training data too precisely

5. **Computational Efficiency**
   - 100x smaller batches → 100x more updates per epoch
   - Faster convergence in practice

### PyTorch Implementation

In [ ]:
# Reset model
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

# SGD: Use small mini-batches
train_loader_sgd = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,  # Small mini-batch
    shuffle=True    # Random sampling adds noise
)

# Optimizer (same as before, but data loading is different)
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01)

# Training loop for SGD
losses_sgd = []

for epoch in range(50):
    for X_batch, y_batch in train_loader_sgd:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward pass (using mini-batch)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer_sgd.zero_grad()
        loss.backward()
        optimizer_sgd.step()
        
        losses_sgd.append(loss.item())

print(f"SGD - Final loss: {losses_sgd[-1]:.4f}")
print(f"SGD made {len(losses_sgd)} updates vs GD's {len(losses_gd)} updates")

### Visualizing SGD Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
ax = axes[0]
ax.plot(losses_gd, linewidth=2, label='Batch GD (Smooth)', alpha=0.8)
ax.plot(losses_sgd, linewidth=1, label='SGD (Noisy)', alpha=0.6)
ax.set_xlabel('Update Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Dynamics: GD vs SGD', fontsize=14, weight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Zoomed view to see noise
ax = axes[1]
ax.plot(losses_sgd[:500], linewidth=1, color='red', alpha=0.6)
ax.set_xlabel('Update Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('SGD Close-up: Notice the Noisy Oscillations', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Observations:")
print("  • Batch GD: Clean, smooth convergence")
print("  • SGD: Noisy, but makes many more updates")
print("  • SGD's noise prevents overfitting to sharp minima!")

### 🔧 Mini-Exercise: Batch Size Impact

**TODO**: Experiment with different batch sizes:

1. Try `batch_size=1` (extreme stochasticity)
2. Try `batch_size=128` (moderate)
3. Try `batch_size=512` (large, closer to Batch GD)

**Observe**:
- How does noise level change?
- Which converges fastest?
- What's the trade-off?

⏱️ ~5 minutes

## 5. Adam: Adaptive Learning with Momentum

### The Motivation

SGD improved on Batch GD, but still has limitations:
- Single learning rate for all parameters
- Sensitive to learning rate choice
- Can oscillate in narrow valleys

**Adam** (Adaptive Moment Estimation) solves these issues.

### How Adam Works

Adam maintains two moving averages:

**1. First Moment (Momentum)**
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

- Accumulates past gradients
- Smooths out noisy gradients
- Accelerates in consistent directions

**2. Second Moment (Adaptive Learning Rate)**
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

- Tracks gradient variance
- Adapts learning rate per parameter
- Large gradients → smaller steps
- Small gradients → larger steps

**3. Parameter Update**
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Where $\hat{m}_t$ and $\hat{v}_t$ are bias-corrected estimates.

### Key Advantages

✅ **Adaptive learning rates**: Each parameter gets its own step size  
✅ **Momentum**: Smooths out SGD's noise while keeping benefits  
✅ **Robust defaults**: Works well with standard settings (lr=0.001, β₁=0.9, β₂=0.999)  
✅ **Fast convergence**: Often faster than SGD  
✅ **Less hyperparameter tuning**: More forgiving than SGD

### PyTorch Implementation

In [ ]:
# Reset model
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

# Same data loader as SGD
train_loader_adam = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

# Adam optimizer
optimizer_adam = optim.Adam(
    model.parameters(),
    lr=0.001,      # Default learning rate for Adam
    betas=(0.9, 0.999),  # (β₁, β₂) - momentum and adaptive rate decay
    eps=1e-8       # Small constant for numerical stability
)

# Training loop for Adam
losses_adam = []

for epoch in range(50):
    for X_batch, y_batch in train_loader_adam:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer_adam.zero_grad()
        loss.backward()
        optimizer_adam.step()  # Adam handles momentum & adaptation internally
        
        losses_adam.append(loss.item())

print(f"Adam - Final loss: {losses_adam[-1]:.4f}")

### Comparing All Three Optimizers

In [ ]:
plt.figure(figsize=(14, 6))

# Plot all three
plt.plot(losses_gd, linewidth=2, label='Batch GD', alpha=0.8)
plt.plot(losses_sgd, linewidth=1, label='SGD', alpha=0.6)
plt.plot(losses_adam, linewidth=2, label='Adam', alpha=0.8)

plt.xlabel('Update Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Optimizer Comparison: GD vs SGD vs Adam', fontsize=14, weight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Summary:")
print(f"  Batch GD: {len(losses_gd)} updates, final loss: {losses_gd[-1]:.4f}")
print(f"  SGD:      {len(losses_sgd)} updates, final loss: {losses_sgd[-1]:.4f}")
print(f"  Adam:     {len(losses_adam)} updates, final loss: {losses_adam[-1]:.4f}")

### 🔧 Mini-Exercise: Adam Hyperparameters

**TODO**: Experiment with Adam's `betas` parameter:

1. Try `betas=(0.5, 0.999)` (less momentum)
2. Try `betas=(0.99, 0.999)` (more momentum)
3. Try `betas=(0.9, 0.99)` (faster adaptation)

**Observe**: How do these affect convergence speed and stability?

⏱️ ~5 minutes

## 6. Practical Comparison on MNIST

Let's compare all three optimizers on a real task: MNIST digit classification.

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

# Create data loaders
train_loader_full = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)  # Batch GD
train_loader_mini = DataLoader(train_dataset, batch_size=64, shuffle=True)  # SGD & Adam
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Define model
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# Training function
def train_model(optimizer_name, train_loader, num_epochs=5):
    model = SimpleMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Create optimizer
    if optimizer_name == 'GD':
        optimizer = optim.SGD(model.parameters(), lr=0.01)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=0.01)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_accuracies = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
        
        # Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                _, predicted = torch.max(outputs.data, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
        
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        
        print(f"{optimizer_name} - Epoch {epoch+1}/{num_epochs}, Test Accuracy: {accuracy:.2f}%")
    
    return train_losses, test_accuracies

In [ ]:
# Train with each optimizer
print("Training with Batch GD...")
losses_gd_mnist, acc_gd = train_model('GD', train_loader_full, num_epochs=5)

print("\nTraining with SGD...")
losses_sgd_mnist, acc_sgd = train_model('SGD', train_loader_mini, num_epochs=5)

print("\nTraining with Adam...")
losses_adam_mnist, acc_adam = train_model('Adam', train_loader_mini, num_epochs=5)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training losses
ax = axes[0]
ax.plot(losses_gd_mnist, label='Batch GD', linewidth=2, alpha=0.8)
ax.plot(losses_sgd_mnist, label='SGD', linewidth=1, alpha=0.6)
ax.plot(losses_adam_mnist, label='Adam', linewidth=2, alpha=0.8)
ax.set_xlabel('Update Step', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Training Loss Comparison on MNIST', fontsize=14, weight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Test accuracies
ax = axes[1]
epochs = range(1, len(acc_gd) + 1)
ax.plot(epochs, acc_gd, 'o-', label='Batch GD', linewidth=2, markersize=8, alpha=0.8)
ax.plot(epochs, acc_sgd, 'o-', label='SGD', linewidth=2, markersize=8, alpha=0.8)
ax.plot(epochs, acc_adam, 'o-', label='Adam', linewidth=2, markersize=8, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy Comparison on MNIST', fontsize=14, weight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🏆 Final Results:")
print(f"  Batch GD: {acc_gd[-1]:.2f}% accuracy")
print(f"  SGD:      {acc_sgd[-1]:.2f}% accuracy")
print(f"  Adam:     {acc_adam[-1]:.2f}% accuracy")

## 7. Summary: Choosing Your Optimizer

### 📊 Quick Comparison

| Optimizer | Convergence | Generalization | Speed | Hyperparameter Sensitivity |
|-----------|-------------|----------------|-------|---------------------------|
| **Batch GD** | Smooth | Poor (sharp minima) | Slow | High |
| **SGD** | Noisy | Good (flat minima) | Fast | High |
| **Adam** | Smooth + Fast | Good | Fast | Low |

### 🎯 When to Use Each

**Batch GD**
- 📚 Learning and understanding optimization
- 🔬 Small datasets only
- ⚠️ Not recommended for production

**SGD**
- 🔬 Research and experimentation
- 🎯 When you need maximum control
- 💪 Training very large models
- ⚡ Often used with momentum: `optim.SGD(lr=0.01, momentum=0.9)`

**Adam**
- 🚀 **Default choice** for most projects
- 🎓 Production systems
- ⚙️ Works well out-of-the-box
- 🎯 When you want fast results with minimal tuning

### 💡 Key Takeaways

1. **Sharp minima = Overfitting**: Batch GD easily falls into these
2. **Noise is beneficial**: SGD's noise prevents overfitting
3. **Flat minima = Better generalization**: SGD and Adam find these
4. **Adam combines the best**: Smooth convergence + good generalization
5. **Start with Adam**: Use `optim.Adam(lr=0.001)` as your default

### 📝 PyTorch Quick Reference

```python
# Batch GD (not recommended)
loader = DataLoader(dataset, batch_size=len(dataset))
optimizer = optim.SGD(model.parameters(), lr=0.01)

# SGD (good for research)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam (recommended default)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

## 🎉 Congratulations!

You now understand:
- ✅ How Batch GD, SGD, and Adam work
- ✅ Why sharp minima cause overfitting
- ✅ How SGD's noise prevents overfitting
- ✅ When to use each optimizer
- ✅ How to implement them in PyTorch

**Next Steps**:
- Learn about learning rate schedules
- Explore other optimizers (AdamW, RMSprop)
- Study gradient clipping and accumulation
- Understand the bias-variance tradeoff in optimization